In [ ]:
import os
import pandas as pd

base_folder = r"C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\training_files\user7"

session_files = [f for f in os.listdir(base_folder) if f.startswith('session')]

session_files.sort()

for file_name in session_files:
    file_path = os.path.join(base_folder, file_name)
    df = pd.read_csv(file_path)  
    print(f"Loaded {file_name}:")
    print(df.head())


Loaded session_0041905381:
   record timestamp  client timestamp    button state    x    y
0               0.0             0.000  NoButton  Move  126  337
1               0.0             0.016  NoButton  Move  129  337
2               0.0             0.032  NoButton  Move  136  337
3               0.0             0.047  NoButton  Move  155  339
4               0.0             0.063  NoButton  Move  185  340
Loaded session_1060325796:
   record timestamp  client timestamp    button state    x    y
0              0.00             0.000  NoButton  Move  374  232
1              0.16             0.016  NoButton  Move  380  240
2              0.16             0.016  NoButton  Move  388  247
3              0.16             0.047  NoButton  Move  392  248
4              0.16             0.172  NoButton  Move  393  248
Loaded session_3320405034:
   record timestamp  client timestamp    button state   x    y
0             0.000             0.000  NoButton  Move  88  551
1             0.264      

In [ ]:
df_cleaned = df.iloc[:, 1:]  # drop record timestamp 

click_mask = (df_cleaned['button'] == 'Left') & (df_cleaned['state'].isin(['Pressed', 'Released']))
dup_mask = df_cleaned.iloc[:, 1:].eq(df_cleaned.iloc[:, 1:].shift()).all(axis=1)
mask = dup_mask & (~click_mask)
df_cleaned = df_cleaned[~mask].reset_index(drop=True)


df_cleaned = df_cleaned[df_cleaned['button'] != 'Scroll'].reset_index(drop=True)


In [3]:


##############################
# SESSION_CUT = 2 # modeling unit - action
##############################
SESSION_CUT = 2
EVAL_TEST_UNIT = 0 #  0: test data are evaluated by session (class probabilities are averaged for all the actions belonging to the test session); 1 -- test data are evaluated by actions (class probability is computed for each action);
NUM_EVAL_ACTIONS = 30 # how many actions are used for decision - only for EVAL_TEST_UNIT = 1

##############################
# ACTIONS settings
##############################
GLOBAL_DELTA_TIME = 10       #DO NOT CHANGE
GLOBAL_MIN_ACTION_LENGTH = 4 #DO NOT CHANGE!!!
GLOBAL_DEBUG = False
CURV_THRESHOLD = 0.0005 # threshold for curvature

# action codes: MM - mouse move; PC - point click; DD - drag and drop
MM = 1
PC = 3
DD = 4

# temporary file
ACTION_CSV_HEADER = "type_of_action,traveled_distance_pixel,elapsed_time,direction_of_movement,straightness,num_points,sum_of_angles,mean_curv,sd_curv,max_curv,min_curv," \
                    "mean_omega,sd_omega,max_omega,min_omega,largest_deviation,dist_end_to_end_line,num_critical_points,"+\
                    "mean_vx,sd_vx,max_vx,min_vx,mean_vy,sd_vy,max_vy,min_vy,mean_v,sd_v,max_v,min_v,mean_a,sd_a,max_a,min_a,mean_jerk,sd_jerk,max_jerk,min_jerk,a_beg_time,n_from,n_to"+\
                    "\n"
ACTION_FILENAME = 'output/balabit_actions.csv'


##############################
# Raw data preprocessing
##############################
# RDP - Remote Desktop Window leaving
X_LIMIT=4000
Y_LIMIT=4000

In [ ]:
import os
import json
import pandas as pd

def get_action_type(button, state):
    if button == 'Left' and state == 'Pressed':
        return 'PointClick'
    if (button == 'Left' and state == 'Released') or (button == 'NoButton' and state == 'Drag'):
        return 'DragDrop'
    if button == 'NoButton' and state == 'Move':
        return 'MouseMove'
    return 'Unknown'

def segment_and_save(df_cleaned, file_path):
    GLOBAL_DELTA_TIME = 10
    GLOBAL_MIN_ACTION_LENGTH = 4

    segments = []
    segment_points = []
    segment_type = None
    segment_start = 0
    prev_time = None

    mousemove_buffer = []

    for i, row in df_cleaned.iterrows():
        current_time = float(row['client timestamp'])
        button = row['button']
        state = row['state']

        current_type = get_action_type(button, state)

        if current_type == 'MouseMove':
            mousemove_buffer.append({
                'x': int(row['x']),
                'y': int(row['y']),
                't': current_time,
                'button': button,
                'state': state
            })

        # When click starts, prepend buffered mousemoves to segment
        if current_type == 'PointClick' and segment_type != 'PointClick':
            # Close previous segment if exists
            if segment_points and len(segment_points) >= GLOBAL_MIN_ACTION_LENGTH:
                segments.append({
                    'type': segment_type,
                    'start_index': segment_start,
                    'end_index': i - 1,
                    'points': [segment_points]
                })

            # Start new segment including buffered mousemoves
            segment_points = mousemove_buffer.copy()
            segment_type = 'PointClick'
            segment_start = i - len(mousemove_buffer)
            if segment_start < 0:
                segment_start = 0
            mousemove_buffer.clear()

        # When segment type changes (other than PointClick start)
        elif current_type != segment_type:
            if segment_points and len(segment_points) >= GLOBAL_MIN_ACTION_LENGTH:
                segments.append({
                    'type': segment_type,
                    'start_index': segment_start,
                    'end_index': i - 1,
                    'points': [segment_points]
                })
            segment_points = []
            segment_type = current_type
            segment_start = i

            # Clear buffer if switching from mousemove
            if current_type != 'MouseMove':
                mousemove_buffer.clear()

        # Add current point unless already added when click started
        if not (current_type == 'PointClick' and segment_type == 'PointClick' and len(segment_points) > 0 and i - segment_start < len(segment_points)):
            segment_points.append({
                'x': int(row['x']),
                'y': int(row['y']),
                't': current_time,
                'button': button,
                'state': state
            })

        prev_time = current_time

    # Add last segment
    if segment_points and len(segment_points) >= GLOBAL_MIN_ACTION_LENGTH:
        segments.append({
            'type': segment_type,
            'start_index': segment_start,
            'end_index': df_cleaned.index[-1],
            'points': [segment_points]
        })

    output = {'segments': segments}

    output_dir = r"C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\training_files\user7\segment"
    os.makedirs(output_dir, exist_ok=True)
    filename = os.path.basename(file_path)
    output_path = os.path.join(output_dir, f"{filename}_segments.json")

    with open(output_path, 'w') as f:
        json.dump(output, f, separators=(',', ':'))

    print(f"Initial segmented file saved to {output_path}")


# Main loop
base_folder = r"C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\training_files\user7"
session_files = [f for f in os.listdir(base_folder) if f.startswith('session')]
session_files.sort()

for file_name in session_files:
    file_path = os.path.join(base_folder, file_name)
    df = pd.read_csv(file_path)

    # Clean duplicates but **keep click rows even if duplicated**
    click_mask = (df['button'] == 'Left') & (df['state'].isin(['Pressed', 'Released']))
    dup_mask = df.iloc[:, 1:].eq(df.iloc[:, 1:].shift()).all(axis=1)
    mask = dup_mask & (~click_mask)
    df_cleaned = df[~mask].reset_index(drop=True)

    # Remove scroll events
    df_cleaned = df_cleaned[df_cleaned['button'] != 'Scroll'].reset_index(drop=True)

    segment_and_save(df_cleaned, file_path)


Initial segmented file saved to C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\training_files\user7\segment\session_0041905381_segments.json
Initial segmented file saved to C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\training_files\user7\segment\session_1060325796_segments.json
Initial segmented file saved to C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\training_files\user7\segment\session_3320405034_segments.json
Initial segmented file saved to C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\training_files\user7\segment\session_3826583375_segments.json
Initial segmented file saved to C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\training_files\user7\segment\session_6668463071_segments.json
Initial segmented file saved to C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\training_files\user7\segment\session_8961330453_segments.json
Initial segmented file saved to C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\training_fil

In [ ]:
import os
import json
import numpy as np
import math
import re

def compute_vx_stats(points):
    x = [float(p['x']) for p in points]
    t = [float(p['t']) for p in points]

    vx = []
    for i in range(1, len(points)):
        dt = t[i] - t[i-1]
        if dt == 0:
            dt = 0.01
        vx.append((x[i] - x[i-1]) / dt)

    if not vx:
        return {'mean_vx': None, 'std_vx': None, 'min_vx': None, 'max_vx': None}

    return {
        'mean_vx': np.mean(vx),
        'std_vx': np.std(vx, ddof=1),
        'min_vx': np.min(vx),
        'max_vx': np.max(vx)
    }

def compute_vy_stats(points):
    y = [float(p['y']) for p in points]
    t = [float(p['t']) for p in points]

    vy = []
    for i in range(1, len(points)):
        dt = t[i] - t[i-1]
        if dt == 0:
            dt = 0.01
        vy.append((y[i] - y[i-1]) / dt)

    if not vy:
        return {'mean_vy': None, 'std_vy': None, 'min_vy': None, 'max_vy': None}

    return {
        'mean_vy': np.mean(vy),
        'std_vy': np.std(vy, ddof=1),
        'min_vy': np.min(vy),
        'max_vy': np.max(vy)
    }

def compute_v_stats(points):
    x = [float(p['x']) for p in points]
    y = [float(p['y']) for p in points]
    t = [float(p['t']) for p in points]

    v = []
    for i in range(1, len(points)):
        dt = t[i] - t[i-1]
        if dt == 0:
            dt = 0.01
        dx = x[i] - x[i-1]
        dy = y[i] - y[i-1]
        v.append(((dx**2 + dy**2)**0.5) / dt)

    if not v:
        return {'mean_v': None, 'std_v': None, 'min_v': None, 'max_v': None}

    return {
        'mean_v': np.mean(v),
        'std_v': np.std(v, ddof=1),
        'min_v': np.min(v),
        'max_v': np.max(v)
    }

def compute_a_stats(points):
    x = [float(p['x']) for p in points]
    y = [float(p['y']) for p in points]
    t = [float(p['t']) for p in points]

    v = []
    for i in range(1, len(points)):
        dt = t[i] - t[i-1]
        if dt == 0:
            dt = 0.01
        dx = x[i] - x[i-1]
        dy = y[i] - y[i-1]
        v.append(((dx**2 + dy**2)**0.5) / dt)

    if len(v) < 2:
        return {'mean_a': None, 'std_a': None, 'min_a': None, 'max_a': None}

    a = []
    for i in range(1, len(v)):
        dt = t[i+1] - t[i]
        if dt == 0:
            dt = 0.01
        dv = v[i] - v[i-1]
        a.append(dv / dt)

    if not a:
        return {'mean_a': None, 'std_a': None, 'min_a': None, 'max_a': None}

    return {
        'mean_a': np.mean(a),
        'std_a': np.std(a, ddof=1),
        'min_a': np.min(a),
        'max_a': np.max(a)
    }

def compute_j_stats(points):
    x = [float(p['x']) for p in points]
    y = [float(p['y']) for p in points]
    t = [float(p['t']) for p in points]

    # Compute velocity v (length: len(points)-1)
    v = []
    for i in range(1, len(points)):
        dt = t[i] - t[i-1]
        if dt == 0:
            dt = 0.01
        dx = x[i] - x[i-1]
        dy = y[i] - y[i-1]
        v.append(((dx**2 + dy**2)**0.5) / dt)

    if len(v) < 2:
        return {'mean_j': None, 'std_j': None, 'min_j': None, 'max_j': None}

    # Compute acceleration a (length: len(v)-1)
    a = []
    for i in range(1, len(v)):
        dt = t[i+1] - t[i]
        if dt == 0:
            dt = 0.01
        dv = v[i] - v[i-1]
        a.append(dv / dt)

    if len(a) < 2:
        return {'mean_j': None, 'std_j': None, 'min_j': None, 'max_j': None}

    # Compute jerk j (length: len(a)-1)
    j = []
    for i in range(1, len(a)):
        dt = t[i+2] - t[i+1]
        if dt == 0:
            dt = 0.01
        da = a[i] - a[i-1]
        j.append(da / dt)

    if not j:
        return {'mean_j': None, 'std_j': None, 'min_j': None, 'max_j': None}

    # Handle std deviation when only one value (avoid NaN)
    std_j = np.std(j, ddof=1) if len(j) > 1 else 0

    return {
        'mean_j': np.mean(j),
        'std_j': std_j,
        'min_j': np.min(j),
        'max_j': np.max(j)
    }


def compute_omega_stats(points):
    t = [float(p['t']) for p in points]
    x = [float(p['x']) for p in points]
    y = [float(p['y']) for p in points]

    angles = [0]
    for i in range(1, len(points)):
        dx = x[i] - x[i-1]
        dy = y[i] - y[i-1]
        angle = math.atan2(dy, dx)
        angles.append(angle)

    omega = []
    for i in range(1, len(angles)):
        dtheta = angles[i] - angles[i-1]
        dt = t[i] - t[i-1]
        if dt == 0:
            dt = 0.01
        omega.append(dtheta / dt)

    if not omega:
        return {'mean_omega': None, 'std_omega': None, 'min_omega': None, 'max_omega': None}

    return {
        'mean_omega': np.mean(omega),
        'std_omega': np.std(omega, ddof=1),
        'min_omega': np.min(omega),
        'max_omega': np.max(omega)
    }

def compute_curvature_stats(points):
    import math
    t = [float(p['t']) for p in points]
    x = [float(p['x']) for p in points]
    y = [float(p['y']) for p in points]

    angles = [0]
    path = [0]
    for i in range(1, len(points)):
        dx = x[i] - x[i-1]
        dy = y[i] - y[i-1]
        distance = math.sqrt(dx*dx + dy*dy)
        path.append(path[-1] + distance)
        angle = math.atan2(dy, dx)
        angles.append(angle)

    curvature = []
    for i in range(1, len(path)):
        dp = path[i] - path[i-1]
        if dp == 0:
            continue
        dangle = angles[i] - angles[i-1]
        curvature.append(dangle / dp)

    if not curvature:
        return {'mean_curv': None, 'std_curv': None, 'min_curv': None, 'max_curv': None}

    return {
        'mean_curv': np.mean(curvature),
        'std_curv': np.std(curvature, ddof=1),
        'min_curv': np.min(curvature),
        'max_curv': np.max(curvature)
    }

def compute_elapsed_time(points):
    t = [float(p['t']) for p in points]
    if not t:
        return {'elapsed_time': None}
    elapsed = t[-1] - t[0]
    return {'elapsed_time': elapsed}

def compute_trajectory_length(points):
    import math
    x = [float(p['x']) for p in points]
    y = [float(p['y']) for p in points]

    if len(points) < 2:
        return {'trajectory_length': 0.0}

    length = 0.0
    for i in range(1, len(points)):
        dx = x[i] - x[i-1]
        dy = y[i] - y[i-1]
        length += math.sqrt(dx*dx + dy*dy)

    return {'trajectory_length': length}


def compute_dist_end_to_end(points):
    import math
    x = float(points[0]['x'])
    y = float(points[0]['y'])
    x_end = float(points[-1]['x'])
    y_end = float(points[-1]['y'])
    
    dist = math.sqrt((x_end - x) ** 2 + (y_end - y) ** 2)
    return {'dist_end_to_end': dist}

def compute_theta(points):
    import math
    x = float(points[-1]['x']) - float(points[0]['x'])
    y = float(points[-1]['y']) - float(points[0]['y'])
    theta = math.atan2(y, x)
    return {'theta': theta}

def compute_direction_from_theta(stats):
    import math
    theta = stats.get('theta')
    if theta is None:
        return {'direction': None}
    if theta < 0:
        theta += 2 * math.pi
    sector = int(theta / (math.pi / 4)) + 1
    if sector > 8:
        sector = 8
    return {'direction': sector}


def compute_straightness(points):
    x = [float(p['x']) for p in points]
    y = [float(p['y']) for p in points]

    # Compute straight-line distance |P1Pn|
    dist_end_to_end = math.sqrt((x[-1] - x[0])**2 + (y[-1] - y[0])**2)

    # Compute trajectory length s
    trajectory_length = 0
    for i in range(1, len(points)):
        dx = x[i] - x[i-1]
        dy = y[i] - y[i-1]
        trajectory_length += math.sqrt(dx*dx + dy*dy)

    if trajectory_length == 0:
        return {'straightness': 0}

    straightness = dist_end_to_end / trajectory_length
    # Clamp straightness to max 1
    if straightness > 1:
        straightness = 1

    return {'straightness': straightness}

def compute_num_points(points):
    return {'num_points': len(points)}

def compute_sum_of_angles(points):
    import math
    x = [float(p['x']) for p in points]
    y = [float(p['y']) for p in points]
    
    if len(points) < 2:
        return {'sum_of_angles': 0}
    
    angles = []
    for i in range(1, len(points)):
        dx = x[i] - x[i-1]
        dy = y[i] - y[i-1]
        angle = math.atan2(dy, dx)
        angles.append(angle)
    
    sum_angles = sum(angles)
    return {'sum_of_angles': sum_angles}

def compute_largest_deviation(points):
    import math

    if len(points) < 3:
        return {'largest_deviation': 0}

    x0, y0 = float(points[0]['x']), float(points[0]['y'])
    xn, yn = float(points[-1]['x']), float(points[-1]['y'])

    # Line coefficients for line through P1 and Pn: ax + by + c = 0
    a = xn - x0
    b = y0 - yn
    c = x0 * yn - xn * y0

    denom = math.sqrt(a**2 + b**2)
    if denom == 0:
        # Start and end points are the same
        return {'largest_deviation': 0}

    max_dev = 0
    for p in points[1:-1]:
        x, y = float(p['x']), float(p['y'])
        dist = abs(a * x + b * y + c) / denom
        if dist > max_dev:
            max_dev = dist

    return {'largest_deviation': max_dev}

def compute_num_sharp_angles(points, TH=math.pi / 4):  # Default TH = 45 degrees
    import math

    if len(points) < 3:
        return {'num_sharp_angles': 0}

    x = [float(p['x']) for p in points]
    y = [float(p['y']) for p in points]

    def angle_between(v1, v2):
        # Compute angle between two vectors v1 and v2
        dot = v1[0]*v2[0] + v1[1]*v2[1]
        mag1 = math.sqrt(v1[0]**2 + v1[1]**2)
        mag2 = math.sqrt(v2[0]**2 + v2[1]**2)
        if mag1 == 0 or mag2 == 0:
            return 0
        cos_theta = dot / (mag1 * mag2)
        # Clamp cos_theta to [-1,1] to avoid numeric errors
        cos_theta = max(min(cos_theta, 1), -1)
        return math.acos(cos_theta)

    count = 0
    for i in range(1, len(points)-1):
        v1 = (x[i] - x[i-1], y[i] - y[i-1])
        v2 = (x[i+1] - x[i], y[i+1] - y[i])
        theta = angle_between(v1, v2)
        if theta < TH:
            count += 1

    return {'num_sharp_angles': count}

def compute_accel_time_at_beginning(points):
    t = [float(p['t']) for p in points]
    x = [float(p['x']) for p in points]
    y = [float(p['y']) for p in points]

    if len(points) < 3:
        return {'a_beg_time': 0.0}

    # Compute velocities
    v = []
    for i in range(1, len(points)):
        dt = t[i] - t[i-1]
        if dt == 0:
            dt = 0.01
        dx = x[i] - x[i-1]
        dy = y[i] - y[i-1]
        v.append(((dx**2 + dy**2)**0.5) / dt)

    # Compute accelerations
    a = []
    for i in range(1, len(v)):
        dt = t[i+1] - t[i]
        if dt == 0:
            dt = 0.01
        dv = v[i] - v[i-1]
        a.append(dv / dt)

    # Sum dt while acceleration is positive from the start
    acc_time = 0.0
    for i, acc in enumerate(a):
        if acc > 0:
            if i == 0:
                acc_time += t[i+2] - t[i+1]  # dt corresponding to acceleration index i
            else:
                acc_time += t[i+1] - t[i]
        else:
            break

    return {'a_beg_time': acc_time}




# List of all feature functions
feature_functions = [
    compute_vx_stats,
    compute_vy_stats,
    compute_v_stats,
    compute_a_stats,
    compute_j_stats,
    compute_omega_stats,
    compute_curvature_stats,
    compute_elapsed_time,
    compute_trajectory_length,
    compute_dist_end_to_end,
    compute_theta,
    compute_direction_from_theta,
    compute_straightness,
    compute_num_points,
    compute_sum_of_angles,
    compute_largest_deviation,
    compute_num_sharp_angles,
    compute_accel_time_at_beginning,
]

segment_folder = r"C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\training_files\user7\segment"
output_folder = r"C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\training_files\user7\features_enhanced"
os.makedirs(output_folder, exist_ok=True)

segment_files = [f for f in os.listdir(segment_folder) if f.endswith('_segments.json')]
segment_files.sort()

for filename in segment_files:
    input_path = os.path.join(segment_folder, filename)
    with open(input_path, 'r') as f:
        data = json.load(f)

    for segment in data['segments']:
        if isinstance(segment['points'], list) and len(segment['points']) == 1 and isinstance(segment['points'][0], list):
            points = segment['points'][0]
        else:
            points = segment['points']

        for func in feature_functions:
            if func == compute_direction_from_theta:
                stats = func(segment)
            elif func == compute_theta:
                stats = func(points)
            else:
                stats = func(points)
            segment.update(stats)

    # Flatten nested points list for final output
    for segment in data['segments']:
        if isinstance(segment['points'], list) and len(segment['points']) == 1 and isinstance(segment['points'][0], list):
            segment['points'] = segment['points'][0]

    output_path = os.path.join(output_folder, filename.replace('_segments.json', '_segments_with_features.json'))
    with open(output_path, 'w') as f:
        json.dump(data, f, separators=(',', ':'))

    # Read back text to format points array nicely with line breaks
    with open(output_path, 'r') as f:
        text = f.read()

    def format_points_inline(match):
        points = match.group(1)
        points = points.replace('},{', '},\n        {')
        return '[\n        ' + points + '\n      ]'

    text = re.sub(r'("segments":\[)', r'\1\n', text)
    text = re.sub(r'"points":(\[[^\]]+\])', lambda m: '"points": ' + format_points_inline(m), text)
    text = re.sub(r'(\})(,)(?=\{"type":)', r'\1\2\n', text)

    with open(output_path, 'w') as f:
        f.write(text)

    print(f"Processed and saved features for {filename} -> {output_path}")

Processed and saved features for session_0041905381_segments.json -> C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\training_files\user7\features_enhanced\session_0041905381_segments_with_features.json
Processed and saved features for session_1060325796_segments.json -> C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\training_files\user7\features_enhanced\session_1060325796_segments_with_features.json
Processed and saved features for session_3320405034_segments.json -> C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\training_files\user7\features_enhanced\session_3320405034_segments_with_features.json
Processed and saved features for session_3826583375_segments.json -> C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\training_files\user7\features_enhanced\session_3826583375_segments_with_features.json
Processed and saved features for session_6668463071_segments.json -> C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\training_files\user7\features_e

In [6]:
import os
import pandas as pd
import json
import re

# Constants used in segmentation
GLOBAL_DELTA_TIME = 10
GLOBAL_MIN_ACTION_LENGTH = 4

# Your feature functions should be defined or imported here
# For brevity, I assume feature_functions, compute_theta, compute_direction_from_theta etc. are defined above

# Your segmentation helper function
def get_action_type(button, state):
    if button == 'Left' and state == 'Pressed':
        return 'PointClick'
    if (button == 'Left' and state == 'Released') or (button == 'NoButton' and state == 'Drag'):
        return 'DragDrop'
    if button == 'NoButton' and state == 'Move':
        return 'MouseMove'
    return 'Unknown'

def segment_and_save(df_cleaned, file_path, segment_folder):
    segments = []
    segment_points = []
    segment_type = None
    segment_start = 0
    prev_time = None

    for i, row in df_cleaned.iterrows():
        current_time = float(row['client timestamp'])
        button = row['button']
        state = row['state']

        current_type = get_action_type(button, state)

        if (segment_type != current_type or
            (prev_time is not None and current_time - prev_time > GLOBAL_DELTA_TIME)):

            if segment_points and len(segment_points) >= GLOBAL_MIN_ACTION_LENGTH:
                segments.append({
                    'type': segment_type,
                    'start_index': segment_start,
                    'end_index': i - 1,
                    'points': [segment_points]  # nested list
                })

            segment_points = []
            segment_type = current_type
            segment_start = i

        point = {
            'x': int(row['x']),
            'y': int(row['y']),
            't': current_time,
            'button': button,
            'state': state
        }
        segment_points.append(point)
        prev_time = current_time

    if segment_points and len(segment_points) >= GLOBAL_MIN_ACTION_LENGTH:
        segments.append({
            'type': segment_type,
            'start_index': segment_start,
            'end_index': df_cleaned.index[-1],
            'points': [segment_points]
        })

    output = {'segments': segments}

    filename = os.path.basename(file_path)
    output_path = os.path.join(segment_folder, f"{filename}_segments.json")

    with open(output_path, 'w') as f:
        json.dump(output, f, separators=(',', ':'))

    print(f"Segmented file saved: {output_path}")

# Feature enhancement function (your provided code)
def add_features_to_segment_file(segment_file_path, output_folder):
    with open(segment_file_path, 'r') as f:
        data = json.load(f)

    for segment in data['segments']:
        if isinstance(segment['points'], list) and len(segment['points']) == 1 and isinstance(segment['points'][0], list):
            points = segment['points'][0]
        else:
            points = segment['points']

        for func in feature_functions:
            if func == compute_direction_from_theta:
                stats = func(segment)
            elif func == compute_theta:
                stats = func(points)
            else:
                stats = func(points)
            segment.update(stats)

    for segment in data['segments']:
        if isinstance(segment['points'], list) and len(segment['points']) == 1 and isinstance(segment['points'][0], list):
            segment['points'] = segment['points'][0]

    filename = os.path.basename(segment_file_path)
    output_path = os.path.join(output_folder, filename.replace('_segments.json', '_segments_with_features.json'))

    with open(output_path, 'w') as f:
        json.dump(data, f, separators=(',', ':'))

    with open(output_path, 'r') as f:
        text = f.read()

    def format_points_inline(match):
        points = match.group(1)
        points = points.replace('},{', '},\n        {')
        return '[\n        ' + points + '\n      ]'

    text = re.sub(r'("segments":\[)', r'\1\n', text)
    text = re.sub(r'"points":(\[[^\]]+\])', lambda m: '"points": ' + format_points_inline(m), text)
    text = re.sub(r'(\})(,)(?=\{"type":)', r'\1\2\n', text)

    with open(output_path, 'w') as f:
        f.write(text)

    print(f"Processed and saved features for {filename} -> {output_path}")

# Main pipeline for test files
base_folder = r"C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\test_files\user7"
segment_folder = os.path.join(base_folder, "segment")
features_folder = os.path.join(base_folder, "features_enhanced")

os.makedirs(segment_folder, exist_ok=True)
os.makedirs(features_folder, exist_ok=True)

session_files = [f for f in os.listdir(base_folder) if f.startswith('session')]
session_files.sort()

for file_name in session_files:
    file_path = os.path.join(base_folder, file_name)
    df = pd.read_csv(file_path)

    # Clean duplicates but keep clicks (same logic as training)
    click_mask = (df['button'] == 'Left') & (df['state'].isin(['Pressed', 'Released']))
    dup_mask = df.iloc[:, 1:].eq(df.iloc[:, 1:].shift()).all(axis=1)
    mask = dup_mask & (~click_mask)
    df_cleaned = df[~mask].reset_index(drop=True)

    df_cleaned = df_cleaned[df_cleaned['button'] != 'Scroll'].reset_index(drop=True)

    # Segment and save JSON files
    segment_and_save(df_cleaned, file_path, segment_folder)

# After segmentation, add features to all segment files
segment_files = [f for f in os.listdir(segment_folder) if f.endswith('_segments.json')]
segment_files.sort()

for seg_file in segment_files:
    seg_path = os.path.join(segment_folder, seg_file)
    add_features_to_segment_file(seg_path, features_folder)


Segmented file saved: C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\test_files\user7\segment\session_0061629194_segments.json
Segmented file saved: C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\test_files\user7\segment\session_0147719489_segments.json
Segmented file saved: C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\test_files\user7\segment\session_0244684556_segments.json
Segmented file saved: C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\test_files\user7\segment\session_0245934723_segments.json
Segmented file saved: C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\test_files\user7\segment\session_0390975032_segments.json
Segmented file saved: C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\test_files\user7\segment\session_0557467514_segments.json
Segmented file saved: C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\test_files\user7\segment\session_0765935758_segments.json
Segmented file saved: C:\Users\ADMIN\Documents\G

In [10]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report, accuracy_score

# Data loading and feature extraction as before
def extract_features_from_file(filepath):
    with open(filepath, 'r') as f:
        data = json.load(f)
    features = []
    for seg in data['segments']:
        feat_vector = []
        for k, v in seg.items():
            if k in ('points', 'type'):
                continue
            feat_vector.append(0.0 if v is None else float(v))
        features.append(feat_vector)
    return np.array(features)

train_folder = r'C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\training_files\user7\features_enhanced'
test_folder = r'C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\test_files\user7\features_enhanced'
labels_df = pd.read_csv(r'C:\Users\ADMIN\Documents\GitHub\Mouseozp\python\balabit\public_labels.csv')

train_files = [f for f in os.listdir(train_folder) if f.endswith('_segments_with_features.json')]
train_X = np.vstack([extract_features_from_file(os.path.join(train_folder, f)) for f in train_files])

def score_test_session(model, session_filename):
    filepath = os.path.join(test_folder, session_filename + '_segments_with_features.json')
    if not os.path.exists(filepath):
        return None
    features = extract_features_from_file(filepath)
    if features.size == 0:
        return None
    if isinstance(model, LocalOutlierFactor):
        # LOF does not have decision_function; use negative_outlier_factor_
        scores = -model._decision_function(features) if hasattr(model, "_decision_function") else -model.negative_outlier_factor_
    else:
        scores = model.decision_function(features)  # positive = inlier, negative = outlier
    return -np.mean(scores)  # flip sign to make higher score = more anomalous

models = {
    'One-Class SVM': OneClassSVM(kernel='rbf', gamma='scale', nu=0.05),
    'Isolation Forest': IsolationForest(contamination=0.05, random_state=42),
    'Local Outlier Factor': LocalOutlierFactor(novelty=True, contamination=0.05)
}

results = {}

for name, model in models.items():
    print(f'Training {name}...')
    if name == 'Local Outlier Factor':
        model.fit(train_X)  # LOF requires novelty=True to use fit_predict on new data
    else:
        model.fit(train_X)

    scores = []
    valid_labels = []
    for _, row in labels_df.iterrows():
        score = score_test_session(model, row['filename'])
        if score is not None:
            scores.append(score)
            valid_labels.append(row['is_illegal'])

    auc = roc_auc_score(valid_labels, scores)

    # Choose threshold as median score or tune as needed; here median used for demonstration
    threshold = np.median(scores)
    preds = [1 if s > threshold else 0 for s in scores]

    cm = confusion_matrix(valid_labels, preds)
    cr = classification_report(valid_labels, preds)
    acc = accuracy_score(valid_labels, preds)

    results[name] = {
        'AUC': auc,
        'Confusion Matrix': cm,
        'Classification Report': cr,
        'Accuracy': acc
    }

for name, metrics in results.items():
    print(f'\n{name} results:')
    print(f"AUC: {metrics['AUC']:.4f}")
    print('Confusion Matrix:')
    print(metrics['Confusion Matrix'])
    print('Classification Report:')
    print(metrics['Classification Report'])
    print(f"Accuracy: {metrics['Accuracy']:.4f}")


Training One-Class SVM...
Training Isolation Forest...
Training Local Outlier Factor...

One-Class SVM results:
AUC: 0.9017
Confusion Matrix:
[[32  4]
 [ 5 32]]
Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.89      0.88        36
           1       0.89      0.86      0.88        37

    accuracy                           0.88        73
   macro avg       0.88      0.88      0.88        73
weighted avg       0.88      0.88      0.88        73

Accuracy: 0.8767

Isolation Forest results:
AUC: 0.9902
Confusion Matrix:
[[35  1]
 [ 2 35]]
Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.97      0.96        36
           1       0.97      0.95      0.96        37

    accuracy                           0.96        73
   macro avg       0.96      0.96      0.96        73
weighted avg       0.96      0.96      0.96        73

Accuracy: 0.9589

Local Outlier Factor results

c:\Users\ADMIN\Documents\GitHub\Mouseozp\.venv-balabit\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\ADMIN\Documents\GitHub\Mouseozp\.venv-balabit\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\ADMIN\Documents\GitHub\Mouseozp\.venv-balabit\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [ ]:
users = ['user7', 'user9', 'user12', 'user15', 'user16', 'user20', 'user21', 'user23', 'user29', 'user35']

all_scores = []
all_labels = []

for user in users:
    # 1. Load user training features (positive)
    user_train_pos = load_features(user)
    
    # 2. Load impostor training features (negative) from other users
    impostor_users = [u for u in users if u != user]
    impostor_train_neg = sample_negatives(impostor_users)
    
    # 3. Balance classes and train classifier (e.g., Random Forest)
    X_train, y_train = balance_and_combine(user_train_pos, impostor_train_neg)
    clf = RandomForestClassifier()
    clf.fit(X_train, y_train)
    
    # 4. Load user test sessions and extract features
    user_test_features, test_labels = load_test_sessions(user)
    
    # 5. Predict scores (probabilities) on test features or aggregated session sets
    scores = clf.predict_proba(user_test_features)[:, 1]
    
    # 6. Collect predictions and labels
    all_scores.extend(scores)
    all_labels.extend(test_labels)

# 7. Evaluate aggregated performance
auc = roc_auc_score(all_labels, all_scores)
print(f'Overall AUC across users: {auc:.4f}')
